# PackageGraph SPARQL Explorer

Query the PackageGraph RDF knowledge graph containing 7M+ triples across 6 Linux distributions and 286K packages.

**Prerequisites:**
- Port-forward Fuseki: `oc port-forward svc/fuseki 3030:3030 -n packagegraph`
- Install dependencies: `pip install sparqlwrapper pandas`

**Ontology namespaces:**
- `pkg:` — Core package concepts (Package, Version, Dependency, Maintainer)
- `sec:` — Security (Vulnerability, CVE, CVSS)
- `vcs:` — Version control (Repository, Commit, Release)
- `met:` — Code metrics (ProgrammingLanguage, CodeMetrics)
- `deb:` / `rpm:` — Distribution-specific extensions
- `slsa:` — SLSA supply chain provenance

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

ENDPOINT = "http://localhost:3030/packagegraph/sparql"

PREFIXES = """
PREFIX pkg:  <https://purl.org/packagegraph/ontology/core#>
PREFIX sec:  <https://purl.org/packagegraph/ontology/security#>
PREFIX vcs:  <https://purl.org/packagegraph/ontology/vcs#>
PREFIX slsa: <https://purl.org/packagegraph/ontology/slsa#>
PREFIX met:  <https://purl.org/packagegraph/ontology/metrics#>
PREFIX deb:  <https://purl.org/packagegraph/ontology/debian#>
PREFIX rpm:  <https://purl.org/packagegraph/ontology/rpm#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
"""

def query(sparql_str: str) -> pd.DataFrame:
    """Execute SPARQL query and return results as a DataFrame."""
    s = SPARQLWrapper(ENDPOINT)
    s.setQuery(PREFIXES + sparql_str)
    s.setReturnFormat(JSON)
    results = s.query().convert()
    bindings = results["results"]["bindings"]
    if not bindings:
        return pd.DataFrame()
    rows = [{k: v["value"] for k, v in b.items()} for b in bindings]
    return pd.DataFrame(rows)

## 1. Dataset Overview

How many triples, and what types of resources exist?

In [ ]:
query("""
SELECT ?type (COUNT(?s) AS ?count) WHERE {
  ?s rdf:type ?type .
}
GROUP BY ?type
ORDER BY DESC(?count)
LIMIT 25
""")

## 2. Distribution Statistics

Packages and versions per distribution.

In [ ]:
query("""
SELECT ?distro (COUNT(DISTINCT ?p) AS ?packages) (COUNT(DISTINCT ?v) AS ?versions) WHERE {
  ?p a pkg:BinaryPackage ;
     pkg:partOfDistribution ?d ;
     pkg:hasVersion ?v .
  ?d rdfs:label ?distro .
}
GROUP BY ?distro
ORDER BY DESC(?packages)
""")

## 3. Search Packages by Name

Find packages matching a pattern across all distributions.

In [ ]:
SEARCH_TERM = "curl"  # Change this

query(f"""
SELECT ?name ?version ?distro ?arch WHERE {{
  ?p a pkg:BinaryPackage ;
     pkg:packageName ?name ;
     pkg:hasVersion ?v ;
     pkg:partOfDistribution ?d .
  ?v pkg:versionString ?version .
  ?d rdfs:label ?distro .
  OPTIONAL {{ ?p pkg:targetArchitecture ?a . ?a rdfs:label ?arch }}
  FILTER(CONTAINS(LCASE(?name), LCASE("{SEARCH_TERM}")))
}}
ORDER BY ?name ?distro
LIMIT 50
""")

## 4. Dependency Graph

What does a specific package depend on?

In [ ]:
PACKAGE = "bash"  # Change this

query(f"""
SELECT ?dep_name ?dep_type WHERE {{
  ?p a pkg:BinaryPackage ;
     pkg:packageName "{PACKAGE}" ;
     pkg:hasDependency ?dep .
  ?dep pkg:dependencyTarget ?target ;
       pkg:dependencyType ?dep_type .
  ?target pkg:packageName ?dep_name .
}}
ORDER BY ?dep_type ?dep_name
""")

## 5. Reverse Dependencies

What packages depend on a given package? ("Who uses this?")

In [ ]:
PACKAGE = "openssl"  # Change this

query(f"""
SELECT ?pkg_name ?dep_type (COUNT(*) AS ?count) WHERE {{
  ?target pkg:packageName "{PACKAGE}" .
  ?dep pkg:dependencyTarget ?target ;
       pkg:dependencyType ?dep_type .
  ?p pkg:hasDependency ?dep ;
     pkg:packageName ?pkg_name .
}}
GROUP BY ?pkg_name ?dep_type
ORDER BY DESC(?count)
LIMIT 30
""")

## 6. Security: Vulnerable Packages

Packages with known CVEs, sorted by recency.

In [ ]:
query("""
SELECT ?pkg_name ?cve_id ?severity ?published WHERE {
  ?vuln a sec:Vulnerability ;
        sec:cveId ?cve_id ;
        sec:affectsVersion ?ver .
  ?pkg pkg:hasVersion ?ver ;
       pkg:packageName ?pkg_name .
  OPTIONAL { ?vuln sec:severity ?severity }
  OPTIONAL { ?vuln sec:publishedDate ?published }
}
ORDER BY DESC(?published)
LIMIT 50
""")

## 7. Security: CVE Details

Detailed information about a specific CVE.

In [ ]:
CVE_ID = "CVE-2022-0778"  # Change this

query(f"""
SELECT ?property ?value WHERE {{
  ?vuln sec:cveId "{CVE_ID}" ;
        ?property ?value .
}}
ORDER BY ?property
""")

## 8. Maintainer Analysis

Who maintains the most packages?

In [ ]:
query("""
SELECT ?name (COUNT(DISTINCT ?p) AS ?packages) WHERE {
  ?p a pkg:BinaryPackage ;
     pkg:maintainedBy ?m .
  ?m foaf:name ?name .
}
GROUP BY ?name
ORDER BY DESC(?packages)
LIMIT 20
""")

## 9. Source Package Mapping

Which source packages produce the most binaries?

In [ ]:
query("""
SELECT ?src_name (COUNT(DISTINCT ?bin) AS ?binaries) WHERE {
  ?bin a pkg:BinaryPackage ;
       pkg:builtFromSource ?src .
  ?src pkg:packageName ?src_name .
}
GROUP BY ?src_name
ORDER BY DESC(?binaries)
LIMIT 20
""")

## 10. Cross-Distribution Comparison

Compare package availability across distributions.

In [ ]:
PACKAGE = "nginx"  # Change this

query(f"""
SELECT ?distro ?version WHERE {{
  ?p a pkg:BinaryPackage ;
     pkg:packageName "{PACKAGE}" ;
     pkg:hasVersion ?v ;
     pkg:partOfDistribution ?d .
  ?v pkg:versionString ?version .
  ?d rdfs:label ?distro .
}}
ORDER BY ?distro
""")

## 11. VCS Repository Metadata

Packages linked to upstream source repositories.

In [ ]:
query("""
SELECT ?pkg_name ?repo_url ?stars ?forks WHERE {
  ?src a pkg:SourcePackage ;
       pkg:packageName ?pkg_name ;
       pkg:upstreamRepository ?repo .
  ?repo vcs:repositoryURL ?repo_url .
  OPTIONAL { ?repo vcs:stargazerCount ?stars }
  OPTIONAL { ?repo vcs:forkCount ?forks }
}
ORDER BY DESC(?stars)
LIMIT 30
""")

## 12. SLSA Provenance Attestations

Packages with build provenance from Koji.

In [ ]:
query("""
SELECT ?pkg_name ?build_level ?builder ?timestamp WHERE {
  ?att a slsa:ProvenanceAttestation ;
       slsa:attestsBuildLevel ?level ;
       slsa:builtBy ?b ;
       slsa:attestationTimestamp ?timestamp .
  ?level rdfs:label ?build_level .
  ?b rdfs:label ?builder .
  ?pkg slsa:hasProvenance ?att ;
       pkg:packageName ?pkg_name .
}
ORDER BY ?pkg_name
LIMIT 30
""")

## 13. Data Freshness

When was each data source last collected?

In [ ]:
query("""
SELECT ?source ?timestamp ?graph WHERE {
  ?snap a pkg:DataSnapshot ;
        pkg:snapshotSource ?source ;
        pkg:snapshotTimestamp ?timestamp .
  OPTIONAL { ?snap pkg:snapshotGraph ?graph }
}
ORDER BY DESC(?timestamp)
""")

## 14. Named Graphs

What named graphs exist and how large are they?

In [ ]:
query("""
SELECT ?graph (COUNT(*) AS ?triples) WHERE {
  GRAPH ?graph { ?s ?p ?o }
}
GROUP BY ?graph
ORDER BY DESC(?triples)
""")

## 15. Custom Query

Write your own SPARQL below. The `query()` helper adds all prefixes automatically.

In [ ]:
query("""
# Your SPARQL query here
SELECT ?s ?p ?o WHERE {
  ?s ?p ?o .
}
LIMIT 10
""")